# Landing → Bronze

Notebook da camada Bronze conforme o escopo do projeto: cria o database, ingere os 5 CSVs sem alterar conteudo/estrutura, adiciona somente ingestion_datetime, grava em Delta com append e ingere a cotacao PTAX do Banco Central.


In [0]:
#Importacao das bibliotecas e configuracoes utilizadas no processo de ingestao
from pyspark.sql import functions as f
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from datetime import datetime, timedelta
import requests

spark.sql("CREATE DATABASE IF NOT EXISTS bronze")

#Definicao do volume de origem dos arquivos CSV
pasta = "/Volumes/workspace/landing/inputs"

#Mapeamento entre os arquivos de entrada e as respectivas tabelas da camada Bronze
mapeamento_oficial = {
    "movies_info_IMDB_TMDB.csv": "bronze.tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "bronze.tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "bronze.tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv": "bronze.tb_credits_and_tags",
    "movies_reviews.csv": "bronze.tb_movies_reviews",
}

#Tratamento de variacao no nome fisico do arquivo sem alterar o nome da tabela de destino
#O destino permanece padronizado como bronze.tb_movies_info
alias_arquivos = {
    "movies_info_IMDB_TMDB.csv": ["movies_info_IMDB_TMDB.csv", "movies_info_TMDB_IMDB.csv"]
}

arquivos_disponiveis = {item.name for item in dbutils.fs.ls(pasta)}

def resolver_arquivo(nome_oficial):
    candidatos = alias_arquivos.get(nome_oficial, [nome_oficial])
    for candidato in candidatos:
        if candidato in arquivos_disponiveis:
            return candidato
    raise FileNotFoundError(
        f"Arquivo obrigatorio nao encontrado em {pasta}: {nome_oficial}. "
        f"Candidatos testados: {candidatos}"
    )

for nome_oficial, tabela in mapeamento_oficial.items():
    print(f"{nome_oficial} -> {resolver_arquivo(nome_oficial)} -> {tabela}")


movies_info_IMDB_TMDB.csv -> movies_info_TMDB_IMDB.csv -> bronze.tb_movies_info
movies_financials_IMDB_TMDB.csv -> movies_financials_IMDB_TMDB.csv -> bronze.tb_movies_financials
movies_metrics_IMDB_TMDB.csv -> movies_metrics_IMDB_TMDB.csv -> bronze.tb_movies_metrics
credits_and_tags_IMDB_TMDB.csv -> credits_and_tags_IMDB_TMDB.csv -> bronze.tb_credits_and_tags
movies_reviews.csv -> movies_reviews.csv -> bronze.tb_movies_reviews


## 1) Ingestao dos cinco CSVs

A Bronze preserva as colunas e os valores dos CSVs. inferSchema=false mantem a leitura bruta como texto. A unica coluna acrescentada e ingestion_datetime.


In [0]:
for nome_oficial, tabela in mapeamento_oficial.items():
    arquivo_real = resolver_arquivo(nome_oficial)
    caminho = f"{pasta}/{arquivo_real}"

    df_bronze = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "false")
        .load(caminho)
        .withColumn("ingestion_datetime", f.current_timestamp())
    )

    #Persistencia em formato Delta utilizando modo append para preservar o historico de ingestao
    (
        df_bronze.write
        .format("delta")
        .mode("append")
        .saveAsTable(tabela)
    )

    print(f"OK: {arquivo_real} -> {tabela}")
    df_bronze.show(5, truncate=False)


OK: movies_info_TMDB_IMDB.csv -> bronze.tb_movies_info
+------+---------+--------------------------+--------------------------+-----------------+------------+-------+--------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------+--------------------------+
|id    |tconst   |title                     |original_title            |original_language|release_date|runtime|status  |overview                                                                                                                                         

## 2) Validacao da Bronze


In [0]:
for tabela in mapeamento_oficial.values():
    df_check = spark.table(tabela)
    assert "ingestion_datetime" in df_check.columns, f"{tabela} sem ingestion_datetime"
    print(f"{tabela}: {df_check.count()} linhas | {len(df_check.columns)} colunas")


bronze.tb_movies_info: 106930 linhas | 11 colunas
bronze.tb_movies_financials: 106165 linhas | 4 colunas
bronze.tb_movies_metrics: 107364 linhas | 7 colunas
bronze.tb_credits_and_tags: 106320 linhas | 10 colunas
bronze.tb_movies_reviews: 32412 linhas | 5 colunas


## 3) Cotacao do dolar — API PTAX do Banco Central

Os widgets usam MM-DD-AAAA. Por padrao, o notebook consulta os ultimos 7 dias corridos ate a data de execucao, mas os valores podem ser sobrescritos na execucao do Job.


In [0]:
#Definicao dinamica do intervalo padrao de consulta considerando os ultimos 7 dias corridos
hoje = datetime.now().date()
data_fim_default = hoje.strftime("%m-%d-%Y")
data_inicio_default = (hoje - timedelta(days=7)).strftime("%m-%d-%Y")

#Criacao dos widgets somente quando necessario para permitir parametrizacao pelo Databricks Workflow
#e preservar valores informados durante a execucao do Job
try:
    dbutils.widgets.get("data_inicio")
except Exception:
    dbutils.widgets.text("data_inicio", data_inicio_default, "Data inicial (MM-DD-AAAA)")

try:
    dbutils.widgets.get("data_fim")
except Exception:
    dbutils.widgets.text("data_fim", data_fim_default, "Data final (MM-DD-AAAA)")

inicio = dbutils.widgets.get("data_inicio").strip()
fim = dbutils.widgets.get("data_fim").strip()

#Validacao do formato e da ordem das datas antes da requisicao a API
dt_inicio = datetime.strptime(inicio, "%m-%d-%Y").date()
dt_fim = datetime.strptime(fim, "%m-%d-%Y").date()
if dt_inicio > dt_fim:
    raise ValueError("data_inicio deve ser menor ou igual a data_fim")

print(f"Periodo PTAX: {inicio} ate {fim}")

url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?"
    f"@dataInicial='{inicio}'&@dataFinalCotacao='{fim}'"
    "&$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

resposta = requests.get(url, timeout=30)
resposta.raise_for_status()
payload = resposta.json()
lista = payload.get("value", [])

if not lista:
    raise ValueError(
        "A API PTAX nao retornou cotacoes no periodo informado. "
        "Use um intervalo que inclua pelo menos um dia util."
    )

schema_ptax = StructType([
    StructField("dataHoraCotacao", StringType(), True),
    StructField("cotacaoCompra", DoubleType(), True),
])

df_dolar = (
    spark.createDataFrame(lista, schema=schema_ptax)
    .withColumn("ingestion_datetime", f.current_timestamp())
)

(
    df_dolar.write
    .format("delta")
    .mode("append")
    .saveAsTable("bronze.tb_cotacao_dolar")
)

df_dolar.orderBy("dataHoraCotacao").show(truncate=False)


Periodo PTAX: 09-14-2026 ate 09-21-2026
+--------------------------+-------------+--------------------------+
|dataHoraCotacao           |cotacaoCompra|ingestion_datetime        |
+--------------------------+-------------+--------------------------+
|2026-09-14 13:10:08.144425|5.169        |2026-09-21 04:03:42.171017|
|2026-09-15 13:09:19.199664|5.1484       |2026-09-21 04:03:42.171017|
|2026-09-16 13:05:30.35873 |5.152        |2026-09-21 04:03:42.171017|
|2026-09-17 13:03:21.858212|5.1515       |2026-09-21 04:03:42.171017|
|2026-09-18 13:03:34.742036|5.1569       |2026-09-21 04:03:42.171017|
+--------------------------+-------------+--------------------------+



## 4) Validacao final


In [0]:
tabelas_esperadas = list(mapeamento_oficial.values()) + ["bronze.tb_cotacao_dolar"]

for tabela in tabelas_esperadas:
    if not spark.catalog.tableExists(tabela):
        raise RuntimeError(f"Tabela nao criada: {tabela}")
    print(f"OK - {tabela}: {spark.table(tabela).count()} linhas")

print("Landing_to_Bronze concluido com sucesso.")


OK - bronze.tb_movies_info: 106930 linhas
OK - bronze.tb_movies_financials: 106165 linhas
OK - bronze.tb_movies_metrics: 107364 linhas
OK - bronze.tb_credits_and_tags: 106320 linhas
OK - bronze.tb_movies_reviews: 32412 linhas
OK - bronze.tb_cotacao_dolar: 5 linhas
Landing_to_Bronze concluido com sucesso.
